## importacao das bibliotecas do ambiente
    basicamente o "requests" permite puxar dados de um servidor
    o "path" é para facilitar a manipulacao de arquivos, tanto no windows quanto em linux
    

#### Objetivo
    objetivo será poder fazer o tratamento desses dados como módulo separado sem ser específicopara a url em geral. Dashboard será específico para esta DB

In [5]:
import requests
from pathlib import Path

salvamos o link da base de dados para caso a url mudar ficar mais fácil pegar novamente

In [6]:
URL_DADOS = (
    "https://s3.sa-east-1.amazonaws.com/"
    "ckan.saude.gov.br/SRAG/2019/"
    "INFLUD19-23-03-2026.csv"
)

antes de baixar os dados e arriscar baixar um arquivo assombrosamente grande verificaremos se o servidor está respondendo

In [7]:
try:
    resposta = requests.get(
        URL_DADOS,
        stream=True,
        timeout=30
    )

    print("Status HTTP:", resposta.status_code)
    print("Tamanho informado (bytes):",
          resposta.headers.get("Content-Length", "Não informado"))

    resposta.raise_for_status()
    print("Conexão estabelecida com sucesso!")

    resposta.close()

except requests.RequestException as erro:
    print("Erro ao acessar a fonte:", erro)

Status HTTP: 200
Tamanho informado (bytes): 51658150
Conexão estabelecida com sucesso!


essa parte vai só verificar se há espaco disponível e se o diretório que será salvo existe

In [8]:
import requests
import shutil
from pathlib import Path

# Localizar a raiz do repositório
raiz = next(
    pasta for pasta in [Path.cwd(), *Path.cwd().parents]
    if (pasta / "requirements.txt").exists()
    and (pasta / "notebooks").is_dir()
)

destino = raiz / "data" / "bronze"
destino.mkdir(parents=True, exist_ok=True)

# Consultar espaço disponível
espaco = shutil.disk_usage(destino)
gib = 1024 ** 3

print(f"Destino: {destino}")
print(f"Espaço livre: {espaco.free / gib:.2f} GiB")

# Consultar tamanho do CSV sem baixar seu conteúdo
resposta = requests.head(
    URL_DADOS,
    allow_redirects=True,
    timeout=30
)

if resposta.ok:
    tamanho = resposta.headers.get("Content-Length")

    if tamanho is not None:
        print(f"Tamanho do arquivo: {int(tamanho) / gib:.3f} GiB")
    else:
        print("Servidor não informou o tamanho.")
else:
    print(f"Consulta de tamanho indisponível: HTTP {resposta.status_code}")

Destino: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV/data/bronze
Espaço livre: 341.08 GiB
Tamanho do arquivo: 0.048 GiB


vai fazer o download em blocos de 1 mb e separa em blocos .part para depois que for concluído juntar tudo.

In [ ]:
import os

# Limites de segurança
MIB = 1024 ** 2
GIB = 1024 ** 3

LIMITE_ARQUIVO = 300 * MIB    # Máximo de 300 MiB por CSV
LIMITE_BRONZE = 2 * GIB       # Máximo de 2 GiB em CSVs Bronze
RESERVA_DISCO = 5 * GIB      # Preservar 5 GiB livres

# Começa desativado para evitar downloads acidentais
BAIXAR = False

arquivo = destino / Path(URL_DADOS).name
temporario = arquivo.with_suffix(".csv.part")

if not BAIXAR:
    print("Download desativado. Nenhum arquivo foi baixado.")

elif arquivo.exists():
    print(f"Arquivo já existe: {arquivo}")
    print("Download ignorado para evitar duplicação.")

else:
    # Espaço já utilizado pelos CSVs Bronze
    uso_bronze = sum(
        p.stat().st_size for p in destino.glob("*.csv")
    )

    if uso_bronze >= LIMITE_BRONZE:
        raise RuntimeError("Limite total da Bronze atingido.")

    try:
        with requests.get(
            URL_DADOS,
            stream=True,
            timeout=(10, 45),
            headers={"Accept-Encoding": "identity"}
        ) as resposta:

            resposta.raise_for_status()

            tamanho_header = resposta.headers.get("Content-Length")
            tamanho_previsto = (
                int(tamanho_header) if tamanho_header else None
            )

            if tamanho_previsto and tamanho_previsto > LIMITE_ARQUIVO:
                raise RuntimeError("Arquivo excede o limite permitido.")

            if tamanho_previsto and uso_bronze + tamanho_previsto > LIMITE_BRONZE:
                raise RuntimeError("Arquivo excederia o limite da Bronze.")

            if temporario.exists():
                temporario.unlink()

            baixados = 0
            ultimo_aviso = 0

            with open(temporario, "wb") as f:
                for bloco in resposta.iter_content(chunk_size=MIB):
                    if not bloco:
                        continue

                    novo_total = baixados + len(bloco)

                    if novo_total > LIMITE_ARQUIVO:
                        raise RuntimeError("Limite do arquivo atingido.")

                    if uso_bronze + novo_total > LIMITE_BRONZE:
                        raise RuntimeError("Limite total da Bronze atingido.")

                    livre = shutil.disk_usage(destino).free

                    if livre - len(bloco) < RESERVA_DISCO:
                        raise RuntimeError("Reserva de disco atingida.")

                    f.write(bloco)
                    baixados = novo_total

                    if baixados - ultimo_aviso >= 50 * MIB:
                        print(f"Baixados: {baixados / MIB:.1f} MiB")
                        ultimo_aviso = baixados

            if tamanho_previsto is not None and baixados != tamanho_previsto:
                raise RuntimeError("Download incompleto: tamanho divergente.")

            # Só confirma o arquivo após concluir a transferência
            os.replace(temporario, arquivo)

            print(f"Download concluído: {arquivo.name}")
            print(f"Tamanho salvo: {baixados / MIB:.2f} MiB")

    finally:
        # Remove arquivo parcial em caso de interrupção ou erro
        temporario.unlink(missing_ok=True)

Download concluído: INFLUD19-23-03-2026.csv
Tamanho salvo: 49.27 MiB


verifica se o arquivo foi salvo corretamente e não houve falhas furante o download (ainda não foi implementada verificacao de integridade)

In [11]:
import hashlib

if arquivo.exists():
    tamanho_real = arquivo.stat().st_size

    with arquivo.open("rb") as f:
        hash_arquivo = hashlib.file_digest(
            f, "sha256"
        ).hexdigest()

    print("Arquivo:", arquivo.name)
    print(f"Tamanho: {tamanho_real / MIB:.2f} MiB")
    print("SHA-256:", hash_arquivo)
    print("Arquivo salvo com sucesso!")
else:
    print("Arquivo não encontrado.")

Arquivo: INFLUD19-23-03-2026.csv
Tamanho: 49.27 MiB
SHA-256: f6de547c1234e2de4d65f830606849a006346e50b32106bd635714665100134a
Arquivo salvo com sucesso!


### verificacao de integridade
    basicamente para verificar se os arquivos nao foram corrompidos durante o tratamento. Não encontrei forma de fazer isso para com a fonte/pós download

In [12]:
import hashlib

if arquivo.exists():
    tamanho_real = arquivo.stat().st_size

    with arquivo.open("rb") as f:
        hash_arquivo = hashlib.file_digest(
            f, "sha256"
        ).hexdigest()

    print("Arquivo:", arquivo.name)
    print(f"Tamanho: {tamanho_real / MIB:.2f} MiB")
    print("SHA-256:", hash_arquivo)
    print("Arquivo salvo com sucesso!")
else:
    print("Arquivo não encontrado.")

Arquivo: INFLUD19-23-03-2026.csv
Tamanho: 49.27 MiB
SHA-256: f6de547c1234e2de4d65f830606849a006346e50b32106bd635714665100134a
Arquivo salvo com sucesso!


#### Verificacao do csv sem le-lo
basicamente isso vai ver se o csv possui conteúdo dentro dele e nao é um arquivo vazio

In [ ]:
import pandas as pd

amostra = pd.read_csv(
    arquivo,
    sep=";",
    encoding="latin-1",
    nrows=5
)

print("Quantidade de colunas:", len(amostra.columns))
print("Colunas encontradas:")
print(amostra.columns.tolist())

contar e validar os registros sem carregar tudo na ram 

In [16]:
import csv

total_registros = 0
linhas_invalidas = 0

with open(arquivo, "r", encoding="latin-1", newline="") as f:
    leitor = csv.reader(f, delimiter=";")

    cabecalho = next(leitor)
    total_colunas = len(cabecalho)

    for linha in leitor:
        total_registros += 1

        if len(linha) != total_colunas:
            linhas_invalidas += 1

print(f"Total de registros: {total_registros:,}")
print(f"Total de colunas: {total_colunas}")
print(f"Linhas com quantidade incorreta de campos: {linhas_invalidas}")

Total de registros: 48,941
Total de colunas: 194
Linhas com quantidade incorreta de campos: 0


registrar os metadados sem copiar os registros em si

In [17]:
import json
from datetime import datetime, timezone

metadados = {
    "ano": 2019,
    "fonte": URL_DADOS,
    "arquivo": arquivo.name,
    "tamanho_bytes": arquivo.stat().st_size,
    "sha256": hash_arquivo,
    "total_registros": total_registros,
    "total_colunas": total_colunas,
    "linhas_invalidas": linhas_invalidas,
    "data_verificacao": datetime.now(timezone.utc).isoformat()
}

# Metadados ficam fora de data/, para poderem ser versionados
pasta_metadados = raiz / "metadata"
pasta_metadados.mkdir(exist_ok=True)

caminho_metadados = pasta_metadados / "bronze_2019.json"

with open(caminho_metadados, "w", encoding="utf-8") as f:
    json.dump(metadados, f, indent=4, ensure_ascii=False)

print(f"Metadados salvos em: {caminho_metadados}")

Metadados salvos em: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV/metadata/bronze_2019.json


#### verificacao inicial da qualidade dos dados 
nao é tratamento ainda essa bosta vai ir pra camada prata

In [ ]:
import pandas as pd

# Contagem de valores vazios por coluna
valores_vazios = pd.Series(dtype="int64")
total_linhas = 0

for bloco in pd.read_csv(
    arquivo,
    sep=";",
    encoding="latin-1",
    dtype=str,
    keep_default_na=False,
    chunksize=5000
):
    vazios = bloco.apply(
        lambda coluna: coluna.str.strip().eq("").sum()
    )

    valores_vazios = valores_vazios.add(
        vazios, fill_value=0
    )

    total_linhas += len(bloco)

diagnostico = pd.DataFrame({
    "Coluna": valores_vazios.index,
    "Valores vazios": valores_vazios.values.astype(int)
})

diagnostico["Percentual vazio"] = (
    diagnostico["Valores vazios"] / total_linhas * 100
).round(2)

diagnostico = diagnostico.sort_values(
    "Percentual vazio",
    ascending=False
)

display(diagnostico.head(20))

basicamente estamos verificando se os campos mais importantes estão chegando vazios por erro na coleta, pois caso haja erro pode influenciar os dados gold

In [20]:
colunas_interesse = [
    "NU_NOTIFIC",
    "DT_NOTIFIC",
    "DT_SIN_PRI",
    "CS_SEXO",
    "SG_UF_NOT",
    "CLASSI_FIN",
    "EVOLUCAO",
    "HOSPITAL",
    "UTI"
]

resumo = (
    diagnostico
    .set_index("Coluna")
    .reindex(colunas_interesse)
    .copy()
)

resumo["Percentual vazio"] = (
    resumo["Valores vazios"] / total_linhas * 100
).round(4)

display(resumo)

,Valores vazios,Percentual vazio
Coluna,,
NU_NOTIFIC,0,0.0000
DT_NOTIFIC,0,0.0000
DT_SIN_PRI,0,0.0000
CS_SEXO,0,0.0000
SG_UF_NOT,42,0.0858
CLASSI_FIN,1026,2.0964
EVOLUCAO,2869,5.8622
HOSPITAL,275,0.5619
UTI,1565,3.1977


salva o resultado da analise para uso posterior,

In [ ]:
import json
from datetime import datetime, timezone

pasta_metadata = raiz / "metadata"
pasta_metadata.mkdir(exist_ok=True)

relatorio = {
    "ano": 2019,
    "arquivo": arquivo.name,
    "total_registros": total_linhas,
    "total_colunas": total_colunas,
    "linhas_invalidas": linhas_invalidas,
    "data_analise": datetime.now(timezone.utc).isoformat(),
    "campos_analisados": {
        coluna: {
            "valores_vazios": int(
                resumo.loc[coluna, "Valores vazios"]
            ),
            "percentual_vazio": float(
                resumo.loc[coluna, "Percentual vazio"]
            )
        }
        for coluna in resumo.index
    }
}

caminho = pasta_metadata / "qualidade_2019.json"

with open(caminho, "w", encoding="utf-8") as f:
    json.dump(relatorio, f, indent=4, ensure_ascii=False)

print("Relatório salvo em:", caminho)

Relatório salvo em: /home/roso/Projetos/R0Z01-Projeto-3-Projeto-integrador---Group-IV/metadata/qualidade_2019.json
